In [1]:
import os
from pathlib import Path

# Walk up from the notebook's location until we find the project root
# (identified by the presence of the 'src' directory)
project_root = Path.cwd()
while not (project_root / 'src').exists() and project_root != project_root.parent:
    project_root = project_root.parent
os.chdir(project_root)
print(os.getcwd())


   StockCode                         Description  \
0      23843         PAPER CRAFT , LITTLE BIRDIE   
1      23166      MEDIUM CERAMIC TOP STORAGE JAR   
2      84077   WORLD WAR 2 GLIDERS ASSTD DESIGNS   
3      22197                      POPCORN HOLDER   
4     85099B             JUMBO BAG RED RETROSPOT   
5     85123A  WHITE HANGING HEART T-LIGHT HOLDER   
6      84879       ASSORTED COLOUR BIRD ORNAMENT   
7      22616          PACK OF 12 LONDON TISSUES    
8      17003                 BROCADE RING PURSE    
9      21212     PACK OF 72 RETROSPOT CAKE CASES   
10     22178     VICTORIAN GLASS HANGING T-LIGHT   
11     21977  PACK OF 60 PINK PAISLEY CAKE CASES   
12     15036           ASSORTED COLOURS SILK FAN   
13     22386             JUMBO BAG PINK POLKADOT   
14     23203            JUMBO BAG VINTAGE DOILY    
15     21915              RED  HARMONICA IN BOX    
16     22469               HEART OF WICKER SMALL   
17     84946        ANTIQUE SILVER T-LIGHT GLASS   
18    85099F

In [5]:
import plotly.express as px

# Sort products by trend change (most negative → most positive)
seasonality_sorted = seasonality.sort_values(
    by='TrendChangeVsMean%',
    ascending=True
)

fig_trends = px.bar(
    seasonality_sorted,
    x='StockCode',
    y='TrendChangeVsMean%',
    color='DataQualityFlag',
    color_discrete_map={
        'OK': '#1f77b4',
        'Low reliability - sparse/spike-driven': '#d62728'
    },
    title='Demand Trend Change (% of Mean) by Product — Flagged Products in Red',
    labels={
        'TrendChangeVsMean%': 'Trend Change (% of historical mean)',
        'StockCode': 'Product'
    }
)

fig_trends.update_layout(
    xaxis_title='Product (StockCode)',
    yaxis_title='Trend Change (% of historical mean)',
    xaxis_tickangle=-45,
    template='plotly_white',
    height=600
)

fig_trends.show()

In [12]:
reliable = seasonality[seasonality['DataQualityFlag'] == 'OK'].copy()
reliable['AbsTrendChange'] = reliable['TrendChangeVsMean%'].abs()
top_3_trend = reliable.sort_values('AbsTrendChange', ascending=False).head(3)['StockCode'].tolist()

print("Top 3 strongest trend changes (reliable) products:", top_3_trend)

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=[f"Product: {code}" for code in top_3_trend]
)

for i, code in enumerate(top_3_trend):
    product_forecast = forecasts[forecasts['StockCode'] == code]
    future = product_forecast[product_forecast['IsFuture'] == True]
    historical = product_forecast[product_forecast['IsFuture'] == False]
    
    fig.add_trace(
        go.Scatter(
            x=historical['ds'], y=historical['yhat'],
            mode='lines', name=f'{code} Historical',
            line=dict(color='blue')
        ), row=i+1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=future['ds'], y=future['yhat'],
            mode='lines', name=f'{code} Forecast',
            line=dict(color='red', dash='dash')
        ), row=i+1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=pd.concat([future['ds'], future['ds'][::-1]]),
            y=pd.concat([future['yhat_upper'], future['yhat_lower'][::-1]]),
            fill='toself',
            fillcolor='rgba(255,0,0,0.1)',
            line=dict(color='rgba(255,255,255,0)'),
            name=f'{code} 95% CI'
        ), row=i+1, col=1
    )

fig.update_layout(
    height=800,
    title_text="Top 3 Strongest Trend Changes (Reliable Only) — Demand Forecast"
)
fig.show()

Top 3 strongest trend changes (reliable) products: ['22086', '23203', '22197']


## Week 4/Day 2: Scale to All Top Products & Seasonality Analysis


**Batch pipeline run (`forecast_pipeline.py`)** — ran Prophet across all 20 products, using the corrected trend-only config from Day 1 (no yearly, no UK holidays). Additional fixes made:

1. **`TrendChangeVsMean%` fix** — the original `TrendChange%` formula divided by `trend_start`, which is often near-zero for low-volume products, producing nonsensical values (e.g. product 22197 showed 28,427.7%). Changed the denominator to the product's historical mean demand instead — a value that's always meaningfully nonzero — giving readable percentages (22197 now shows 207.7%).

2. **Data quality flagging (`assess_forecastability`)** — added a check to flag products where the demand pattern is too sparse/spike-driven for trend + seasonality to produce a meaningful forecast (single huge bulk orders surrounded by mostly-zero weeks — likely wholesale/reseller buyers in this dataset). Flag requires **both** >40% zero-demand weeks **and** a >15x spike-to-mean ratio — tested with OR first, which incorrectly flagged product 23203 (only failed one condition, forecast looked fine); switched to AND after confirming 23203's forecast was legitimate. Correctly flags only 23843 and 23166, which show genuinely broken output (implausible flat/negative forecasts) even after the trend-only fix.

3. **Weekly seasonality fix (mid-Day 2, before batch re-run)** — discovered that Day 1's assumption about weekly seasonality being trustworthy was wrong (see correction note in Day 1 notebook). Disabled `weekly_seasonality` and switched forecast frequency to `freq='W-MON'` to match the training data's weekly cadence. Re-ran the full batch after this fix — see below for validation.

**Data loading cell** — reloaded `all_forecasts.csv` and `seasonality_analysis.csv` after the weekly seasonality fix, confirming the same 20 products and same 2 data-quality flags as before the fix (23843, 23166) — the flagging logic is independent of Prophet's seasonality settings, so it correctly held steady.

**Trend bar chart** — plotted `TrendChangeVsMean%` per product, colored by data quality flag. The two flagged products are clear outliers (655% and -463%), visually separated from all "OK" products (which range roughly -55% to +370%).

**Top 3 strongest trend changes plot** — filtered to `DataQualityFlag == 'OK'` only, then selected the 3 products with the largest absolute `TrendChangeVsMean%`: 22086, 23203, 22197. All three show a clean, continuous linear trend from near-zero in early 2011 up to their historical peak by Nov/Dec 2011, with the forecast simply continuing that same line — no discontinuity at the historical/forecast boundary. This is the expected behavior of a trend-only model and confirms the weekly seasonality fix worked: compare to product 84077 pre-fix, which crashed from a 
~910 historical mean to a ~100 forecast right at that same boundary.

### Why "seasonality" isn't the right frame anymore

The original Week 4 plan's Day 2 goal was "multi-product forecasts and seasonality analysis." Since yearly, weekly, and holiday seasonality are all disabled (see Day 1 + this notebook), there is no seasonality component left in these models — only trend. Renamed the "most seasonal products" comparison to "strongest trend changes" to accurately describe what's actually being compared. This is a direct consequence of the dataset's structure (only ~1 year of history, weekly aggregation that collapsed to a single weekday) rather than a modeling choice — worth stating clearly in the README later as a documented limitation, alongside a note that daily-level re-aggregation from the raw transaction data would be needed to recover real day-of-week or holiday effects.

### Validation: 84077 before vs. after the weekly seasonality fix

| | Historical mean | Forecast (pre-fix) | Forecast (post-fix) |
|---|---|---|---|
| Product 84077 | 910.8 | 96 – 118 | 996 – 1018 |

The post-fix forecast sits almost exactly at the historical mean, as expected for a stable trend-only model — strong confirmation the fix addressed the actual bug rather than just moving the numbers around.